
# 01: Fetch Abstracts for Referenced Studies

## Objective
Match our extracted references to PubMed records and fetch abstracts for LLM evaluation.

## Matching Strategy (CrossRef-enhanced with PubMed fallback)

This workflow uses a multi-phase approach for maximum match rate:

1. **Direct Extraction** — Extract DOI/PMID directly from reference text using regex
2. **CrossRef API** — For references without DOI, query CrossRef's bibliographic search
3. **PubMed Lookup** — Use DOIs to fetch PMIDs, then batch-fetch abstracts
4. **CrossRef Abstract Fallback** — For papers with DOI but no PubMed abstract
5. **PubMed Direct Title Search** — For papers still without abstracts, search PubMed
   directly by title with fuzzy matching to catch papers missed by DOI-based lookups

## Why CrossRef?
- CrossRef API (`query.bibliographic`) is specifically designed to match reference strings to DOIs
- Handles fuzzy matching internally (typos, ligatures, formatting issues)
- Has 130M+ DOIs - broader coverage than PubMed-only search
- DOI → PMID lookup is highly reliable (~95%+ when the paper exists in PubMed)

## Why PubMed Direct Title Search?
- Some papers exist in PubMed but lack a DOI link (especially older papers)
- CrossRef may find a DOI but PubMed doesn't index that DOI, so DOI→PMID fails
- Direct title search with fuzzy validation catches these edge cases

## Output
- `Data/referenced_paper_abstracts.csv` - Abstracts with match confidence scores


---
## 1. Environment Setup

In [2]:
%pip install -q biopython pandas tqdm requests rapidfuzz

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Imports and configuration for reference matching pipeline.
# Sets up NCBI Entrez API credentials (loaded from environment variables for security),
# defines rate limits for API calls, and configures fuzzy matching thresholds
# for validating CrossRef and PubMed search results.

# --- Standard library imports ---
import re
import time
import os
import unicodedata
from pathlib import Path
from collections import Counter

# --- Third-party imports ---
import pandas as pd
import numpy as np
import requests
from tqdm.notebook import tqdm
from Bio import Entrez                    # NCBI API client
from rapidfuzz import fuzz                # Fast fuzzy string matching

# --- Configure NCBI Entrez API ---
# Credentials loaded from environment variables for security
# API key increases rate limit from 3/sec to 10/sec
Entrez.email = os.environ.get("NCBI_EMAIL", "")
Entrez.api_key = os.environ.get("NCBI_API_KEY", "")

# --- Define API rate limits (seconds between calls) ---
NCBI_RATE = 0.11 if Entrez.api_key else 0.34  # With/without API key
CROSSREF_RATE = 0.5                            # CrossRef polite pool

# --- Configure fuzzy matching thresholds ---
# These control how strictly we validate search results
FUZZY_TITLE_THRESHOLD = 75    # Minimum title similarity score (0-100)
FUZZY_AUTHOR_MIN = 30         # Minimum author similarity score
FUZZY_AUTHOR_WEIGHT = 0.3     # Weight for author score in combined scoring

# --- Determine project paths ---
notebook_dir = Path.cwd()
project_root = notebook_dir if (notebook_dir / "Data").exists() else notebook_dir.parent
DATA_DIR = project_root / "Data"

# --- Define input/output file paths ---
REFS_CSV = DATA_DIR / "categorized_references.csv"    # From NB00
META_CSV = DATA_DIR / "review_metadata.csv"           # From NB00

OUTPUT_CSV = DATA_DIR / "referenced_paper_abstracts.csv"  # Final output
PROGRESS_CSV = DATA_DIR / "crossref_matching_progress.csv"  # Checkpoint for resumption

# --- Display configuration summary ---
print(f"Data directory: {DATA_DIR}")
print(f"NCBI API key configured: {bool(Entrez.api_key)}")
print(f"NCBI rate: {1/NCBI_RATE:.0f} requests/sec")
print(f"Fuzzy match threshold: {FUZZY_TITLE_THRESHOLD}% title, {FUZZY_AUTHOR_MIN}% author (when provided)")

Data directory: c:\Users\juanx\Documents\LSE-UKHSA Project\Data
NCBI API key configured: True
NCBI rate: 9 requests/sec
Fuzzy match threshold: 75% title, 30% author (when provided)


---
## 2. Load and Prepare Data
Load references from NB00 and create deduplicated working set.

In [ ]:
# Load references and metadata CSV files.
# The references were extracted from 119 public health Cochrane PDFs in notebook 00.

# ─────────────────────────────────────────────────────────────────────────────
# Load the categorized references extracted from Cochrane review PDFs
# This file contains all references cited in the reviews (included, excluded, awaiting, ongoing)
# ─────────────────────────────────────────────────────────────────────────────
refs_df = pd.read_csv(REFS_CSV)
print(f"Total references: {len(refs_df):,}")

# ─────────────────────────────────────────────────────────────────────────────
# Load review metadata (titles, abstracts, DOIs) for linking references to reviews
# ─────────────────────────────────────────────────────────────────────────────
meta_df = pd.read_csv(META_CSV)
print(f"Total reviews in metadata: {len(meta_df):,}")

# ─────────────────────────────────────────────────────────────────────────────
# Display category breakdown (included, excluded, awaiting, ongoing studies)
# ─────────────────────────────────────────────────────────────────────────────
print(f"\nUnique reviews: {refs_df['review_doi'].nunique():,}")
print(f"Categories:\n{refs_df['category'].value_counts().to_string()}")

Total references: 17,368
Total reviews in metadata: 119

Unique reviews: 113
Categories:
category
excluded          6910
additional        5704
included          3983
awaiting           459
ongoing            234
other_versions      78


In [ ]:
# Deduplicate references using a normalized signature.
# Creates unique identifiers from first author, year, and title prefix to avoid
# processing the same reference multiple times across different reviews.

# ─────────────────────────────────────────────────────────────────────────────
# Define signature function: combines first author, year, and title prefix
# This creates a unique fingerprint for each reference regardless of formatting
# ─────────────────────────────────────────────────────────────────────────────
def create_ref_signature(row):
    """Create a normalized signature for deduplication."""
    # Extract and normalize the title (lowercase, trimmed to 100 chars)
    title = str(row.get('title', '')).lower().strip()[:100]
    
    # Extract publication year
    year = str(row.get('year', ''))
    
    # Extract first author's surname (first word of authors string)
    first_author = str(row.get('authors', '')).split()[0].lower() if row.get('authors') else ''
    
    # Combine into a unique signature using first 50 chars of title
    return f"{first_author}|{year}|{title[:50]}"

# ─────────────────────────────────────────────────────────────────────────────
# Apply deduplication: same reference may appear in multiple reviews
# We only need to process each unique reference once for API lookups
# ─────────────────────────────────────────────────────────────────────────────
refs_df['signature'] = refs_df.apply(create_ref_signature, axis=1)
unique_refs = refs_df.drop_duplicates(subset='signature').copy()

# ─────────────────────────────────────────────────────────────────────────────
# Report deduplication results
# ─────────────────────────────────────────────────────────────────────────────
print(f"Total references: {len(refs_df):,}")
print(f"Unique references: {len(unique_refs):,}")
print(f"Deduplication ratio: {len(refs_df)/len(unique_refs):.1f}x")
print(f"\nCategories:")
print(unique_refs['category'].value_counts().to_string())

Total references: 17,368
Unique references: 15,712
Deduplication ratio: 1.1x

Categories:
category
excluded          6468
additional        4974
included          3534
awaiting           446
ongoing            219
other_versions      71


---
## 3. Phase 1: Direct Identifier Extraction
Extract DOIs and PMIDs directly from reference text using regex patterns.

In [ ]:
# Phase 1: Extract DOI and PMID identifiers directly from reference text.
# Many references contain embedded identifiers that can be extracted using
# regex patterns, avoiding the need for external API lookups.

# ─────────────────────────────────────────────────────────────────────────────
# PMID extraction function: looks for patterns like "PMID: 12345678"
# Common formats: "PMID:12345", "PUBMED: 12345", "MEDLINE: 12345"
# ─────────────────────────────────────────────────────────────────────────────
def extract_pmid(text):
    """Extract PMID from reference text."""
    if pd.isna(text):
        return None
    # Match PMID/PUBMED/MEDLINE followed by optional colon/space and digits
    match = re.search(r"(PMID|PUBMED|MEDLINE)[:\s]*(\d+)", str(text), flags=re.I)
    if match:
        return match.group(2)  # Return just the numeric ID
    return None

# ─────────────────────────────────────────────────────────────────────────────
# DOI extraction function: looks for patterns like "10.1234/example.doi"
# Handles URLs (https://doi.org/...) and raw DOIs with "DOI:" prefix
# ─────────────────────────────────────────────────────────────────────────────
def extract_doi(text):
    """Extract DOI from reference text."""
    if pd.isna(text):
        return None
    # Match DOI format: 10.XXXX/... (with optional DOI: prefix or URL)
    match = re.search(
        r"(?:DOI[:\s]*|HTTPS?://(?:DX\.)?DOI\.ORG/)?(10\.\d{4,9}/[-._;()/:A-Z0-9]+)", 
        str(text), flags=re.I
    )
    if match:
        doi = match.group(1)
        return doi.rstrip(".,;)")  # Clean trailing punctuation from DOI
    return None

# ─────────────────────────────────────────────────────────────────────────────
# Combine title, authors, year into a single searchable text field
# This gives us maximum chance to find embedded identifiers
# ─────────────────────────────────────────────────────────────────────────────
unique_refs['full_ref'] = (
    unique_refs['title'].fillna('') + ' ' + 
    unique_refs['authors'].fillna('') + ' ' +
    unique_refs['year'].fillna('').astype(str)
)

# ─────────────────────────────────────────────────────────────────────────────
# Apply extraction functions to all references
# ─────────────────────────────────────────────────────────────────────────────
unique_refs['extracted_doi'] = unique_refs['full_ref'].apply(extract_doi)
unique_refs['extracted_pmid'] = unique_refs['full_ref'].apply(extract_pmid)

# ─────────────────────────────────────────────────────────────────────────────
# Merge with any pre-existing identifiers from reference metadata
# Use extracted values as primary, fall back to existing if available
# ─────────────────────────────────────────────────────────────────────────────
unique_refs['final_doi'] = unique_refs['extracted_doi'].combine_first(unique_refs.get('ref_doi'))
unique_refs['final_pmid'] = unique_refs['extracted_pmid'].combine_first(unique_refs.get('pmid'))

# ─────────────────────────────────────────────────────────────────────────────
# Report Phase 1 results: how many identifiers extracted without API calls
# ─────────────────────────────────────────────────────────────────────────────
has_doi = unique_refs['final_doi'].notna().sum()
has_pmid = unique_refs['final_pmid'].notna().sum()

print("PHASE 1: Direct Extraction")
print("=" * 60)
print(f"References with DOI: {has_doi:,} ({has_doi/len(unique_refs)*100:.1f}%)")
print(f"References with PMID: {has_pmid:,} ({has_pmid/len(unique_refs)*100:.1f}%)")
print(f"Need CrossRef lookup: {len(unique_refs) - has_doi:,}")

PHASE 1: Direct Extraction
References with DOI: 738 (4.7%)
References with PMID: 103 (0.7%)
Need CrossRef lookup: 14,974


C:\Users\juanx\AppData\Local\Temp\ipykernel_9288\1551778175.py:41: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  unique_refs['final_pmid'] = unique_refs['extracted_pmid'].combine_first(unique_refs.get('pmid'))


---
## 4. API Functions
Core functions for CrossRef DOI lookup, PubMed PMID conversion, abstract fetching, and fuzzy matching validation.

In [ ]:
# CrossRef and PubMed API functions for reference matching.
# Includes text normalization utilities for handling PDF ligatures, fuzzy matching
# functions for validating search results, CrossRef DOI lookup with short-title
# fallback, PubMed abstract fetching, and direct title search capabilities.

# =============================================================================
# TEXT NORMALIZATION UTILITIES
# PDF extraction often produces Unicode ligatures (e.g., "ﬁ" instead of "fi")
# and special characters that need normalization for accurate matching
# =============================================================================

def normalize_text(text):
    """Normalize unicode ligatures and special characters from PDFs."""
    if not text:
        return ""
    # NFKD normalization decomposes ligatures like "ﬁ" → "fi"
    text = unicodedata.normalize("NFKD", str(text))
    return text


def clean_text_for_matching(text):
    """Clean text for fuzzy matching comparison."""
    if not text:
        return ""
    text = normalize_text(text)
    # Remove all punctuation except hyphens, convert to lowercase
    text = re.sub(r"[^\w\s\-]", "", text.lower())
    return text.strip()


def clean_reference(ref):
    """Clean reference text for CrossRef query."""
    if pd.isna(ref):
        return ""
    ref = str(ref)
    ref = normalize_text(ref)
    # Remove Cochrane boilerplate text that pollutes search queries
    ref = re.sub(r"\(REVIEW\).*", "", ref, flags=re.I)
    ref = re.sub(r"TRUSTED EVIDENCE.*", "", ref, flags=re.I)
    ref = re.sub(r"COCHRANE.*?LIBRARY", "", ref, flags=re.I)
    # Remove page markers from PDF extraction
    ref = re.sub(r"---\s*Page\s*\d+\s*---", " ", ref)
    # Collapse multiple whitespace
    ref = re.sub(r"\s{2,}", " ", ref)
    return ref.strip()


def clean_title_for_search(title):
    """Normalize title for CrossRef API search."""
    title = normalize_text(title)
    # Replace slashes with spaces (common in titles)
    title = title.replace("/", " ")
    # Collapse whitespace
    title = re.sub(r"\s+", " ", title).strip()
    return title


def extract_jats_abstract(text):
    """Strip JATS/HTML tags from CrossRef abstract text."""
    if not text:
        return None
    # CrossRef abstracts may contain JATS XML tags - strip them
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text if text else None


# =============================================================================
# CROSSREF API FUNCTIONS
# CrossRef provides bibliographic metadata including DOIs for scholarly works
# =============================================================================

def format_crossref_authors(work):
    """Format author names from CrossRef response."""
    authors = work.get("author", [])
    parts = []
    for a in authors:
        # Extract family name and given name initials
        last = a.get("family", "")
        given = a.get("given", "")
        initials = "".join(w[0] for w in given.split()) if given else ""
        parts.append(f"{last} {initials}".strip())
    return ", ".join(parts) if parts else None


def score_crossref_candidates(candidates, query_title, query_authors):
    """Score CrossRef candidates using fuzzy matching."""
    clean_query_title = clean_text_for_matching(query_title)
    clean_query_authors = clean_text_for_matching(query_authors)

    scored = []
    for item in candidates:
        # Get title from CrossRef response (stored as array)
        cr_titles = item.get("title", [])
        cr_title = cr_titles[0] if cr_titles else ""

        # Calculate fuzzy match score for title (0-100)
        title_score = fuzz.token_sort_ratio(clean_query_title, clean_text_for_matching(cr_title))

        # Calculate fuzzy match score for authors (0-100)
        cr_authors = format_crossref_authors(item) or ""
        author_score = fuzz.token_sort_ratio(clean_query_authors, clean_text_for_matching(cr_authors))

        scored.append((item, title_score, author_score))

    # Filter to candidates meeting threshold requirements
    above_threshold = [
        (item, ts, aus) for item, ts, aus in scored 
        if ts >= FUZZY_TITLE_THRESHOLD and (
            not clean_query_authors  # Skip author check if no authors provided
            or aus >= FUZZY_AUTHOR_MIN
        )
    ]

    if not above_threshold:
        return None, None, None

    # Return best match (highest title score, then author score as tiebreaker)
    best = max(above_threshold, key=lambda x: (x[1], x[2]))
    return best


def get_doi_from_crossref(ref, title=None, authors=None, timeout=(5, 30)):
    """Query CrossRef API to get DOI from bibliographic reference with fuzzy matching."""
    # Legacy mode: simple lookup without fuzzy matching (for backward compatibility)
    legacy_mode = (title is None and authors is None)

    # Skip very short references (likely incomplete)
    if not ref or len(ref) < 20:
        return None if legacy_mode else (None, None, None, None)

    # Set polite User-Agent header (recommended by CrossRef)
    headers = {
        "User-Agent": f"LSE-UKHSA-Project/1.0 (mailto:{Entrez.email})" if Entrez.email else "LSE-UKHSA-Project/1.0"
    }

    if title is None:
        title = ref
    if authors is None:
        authors = ""

    clean_title = clean_title_for_search(title)

    def try_search(search_term, method_name):
        """Helper to search CrossRef and score results."""
        try:
            url = "https://api.crossref.org/works"
            params = {
                "query.bibliographic": search_term,  # Searches across all bibliographic fields
                "rows": 10,  # Return top 10 candidates for scoring
                "select": "DOI,title,author,abstract"  # Only fetch needed fields
            }
            r = requests.get(url, params=params, headers=headers, timeout=timeout)
            r.raise_for_status()

            items = r.json().get("message", {}).get("items", [])
            if not items:
                return None, None, None, None

            # In legacy mode, just return first result
            if legacy_mode:
                return items[0].get("DOI"), None, None, None

            # Score candidates against query using fuzzy matching
            best_item, title_score, author_score = score_crossref_candidates(items, title, authors)

            if best_item:
                return best_item.get("DOI"), method_name, title_score, author_score

            # Check for exact title match (case-insensitive) as fallback
            first_title = (items[0].get("title") or [""])[0]
            if clean_text_for_matching(first_title) == clean_text_for_matching(title):
                return items[0].get("DOI"), f"{method_name}_exact", 100, None

        except Exception:
            pass
        return None, None, None, None

    # STRATEGY 1: Search with full reference text
    doi, method, ts, aus = try_search(ref, "crossref_fuzzy")
    if doi:
        return doi if legacy_mode else (doi, method, ts, aus)

    # STRATEGY 2: Search with cleaned title only
    doi, method, ts, aus = try_search(clean_title, "crossref_title")
    if doi:
        return doi if legacy_mode else (doi, method, ts, aus)

    # STRATEGY 3: Search with short title (before colon/hyphen)
    # Helps with subtitles: "Main Title: A Subtitle" → "Main Title"
    short_title = re.split(r"\s*[:\-]\s*", clean_title)[0].strip()
    if short_title.lower() != clean_title.lower() and len(short_title) > 20:
        doi, method, ts, aus = try_search(short_title, "crossref_short")
        if doi:
            return doi if legacy_mode else (doi, method, ts, aus)

    return None if legacy_mode else (None, "no_match", None, None)


def fetch_crossref_abstract(doi, timeout=(5, 30)):
    """Fetch abstract from CrossRef for a given DOI."""
    if not doi:
        return None

    headers = {
        "User-Agent": f"LSE-UKHSA-Project/1.0 (mailto:{Entrez.email})" if Entrez.email else "LSE-UKHSA-Project/1.0"
    }

    try:
        # Direct DOI lookup endpoint
        url = f"https://api.crossref.org/works/{doi.strip()}"
        r = requests.get(url, headers=headers, timeout=timeout)
        if r.status_code == 200:
            work = r.json().get("message", {})
            raw_abstract = work.get("abstract", "")
            # Clean JATS XML tags from abstract
            return extract_jats_abstract(raw_abstract)
    except Exception:
        pass
    return None


# =============================================================================
# PUBMED API FUNCTIONS
# PubMed provides medical literature data including abstracts via NCBI Entrez
# =============================================================================

def get_pmid_from_doi(doi, api_key=None):
    """Look up PMID from DOI via PubMed."""
    if not doi:
        return None

    try:
        params = {
            "db": "pubmed",
            "term": f"{doi}[DOI]",  # Search DOI field
            "retmode": "json"
        }
        if api_key:
            params["api_key"] = api_key

        r = requests.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
            params=params, timeout=(5, 30)
        )
        data = r.json()
        idlist = data.get("esearchresult", {}).get("idlist", [])
        return idlist[0] if idlist else None
    except Exception:
        return None


def fetch_abstracts_batch(pmids, batch_size=200, max_retries=3):
    """Batch fetch PubMed records with retry logic."""
    results = {}
    # Convert to list of valid numeric PMIDs
    pmid_list = [str(int(p)) for p in pmids if pd.notna(p) and str(p).isdigit()]
    failed_batches = []

    # Process in batches to avoid API limits
    for i in range(0, len(pmid_list), batch_size):
        batch = pmid_list[i:i + batch_size]
        success = False

        # Retry logic for transient failures
        for attempt in range(max_retries):
            try:
                # Rate limiting: wait longer on retries
                time.sleep(NCBI_RATE * (attempt + 1))
                
                # Fetch batch via Entrez efetch
                handle = Entrez.efetch(db="pubmed", id=",".join(batch), rettype="xml", retmode="xml")
                records = Entrez.read(handle)
                handle.close()

                # Extract data from each article
                for article in records.get('PubmedArticle', []):
                    data = extract_record_data(article)
                    if data:
                        results[data['pmid']] = data
                success = True
                break
            except Exception as e:
                if attempt < max_retries - 1:
                    print(f"Batch {i//batch_size + 1} attempt {attempt + 1} failed: {e}, retrying...")
                else:
                    print(f"Batch {i//batch_size + 1} failed after {max_retries} attempts: {e}")
                    failed_batches.append(batch)

        if not success:
            failed_batches.append(batch)

    return results, failed_batches


def extract_record_data(record):
    """Extract fields from PubMed record."""
    try:
        article = record['MedlineCitation']['Article']
        pmid = str(record['MedlineCitation']['PMID'])
        title = str(article.get('ArticleTitle', ''))

        # Extract abstract (may be structured with multiple parts)
        abstract = ''
        if 'Abstract' in article and 'AbstractText' in article['Abstract']:
            parts = article['Abstract']['AbstractText']
            # Join multiple abstract parts (Background, Methods, Results, etc.)
            abstract = ' '.join([str(p) for p in parts]) if isinstance(parts, list) else str(parts)

        # Extract publication year from journal info
        year = ''
        if 'Journal' in article and 'JournalIssue' in article['Journal']:
            year = article['Journal']['JournalIssue'].get('PubDate', {}).get('Year', '')

        # Extract author list as "LastName Initials" format
        authors = []
        if 'AuthorList' in article:
            for auth in article['AuthorList']:
                if 'LastName' in auth:
                    name = auth['LastName'] + (' ' + auth.get('Initials', ''))
                    authors.append(name.strip())

        # Extract DOI from ELocationID field
        doi = ''
        if 'ELocationID' in article:
            for loc in article['ELocationID']:
                if loc.attributes.get('EIdType') == 'doi':
                    doi = str(loc)
                    break

        return {
            'pmid': pmid, 'title': title, 'abstract': abstract,
            'year': year, 'authors': '; '.join(authors), 'doi': doi
        }
    except:
        return None


def search_pubmed_by_title(title, authors=None, year=None):
    """Search PubMed directly by title with fuzzy matching validation."""
    if not title or not isinstance(title, str) or len(title) < 10:
        return None, None

    clean_title = clean_title_for_search(title)

    # Build query list: first try with year filter, then without
    queries = [clean_title]
    if year and str(year).isdigit():
        queries.insert(0, f'{clean_title} AND {year}[PDAT]')

    for query in queries:
        try:
            time.sleep(NCBI_RATE)

            # Search PubMed by query
            params = {
                "db": "pubmed",
                "term": query,
                "retmax": 10,  # Get top 10 candidates for fuzzy matching
                "retmode": "json",
            }
            if Entrez.api_key:
                params["api_key"] = Entrez.api_key
            r = requests.get(
                "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
                params=params, timeout=(5, 15)
            )
            idlist = r.json().get("esearchresult", {}).get("idlist", [])
            if not idlist:
                continue

            # Fetch full records for candidates
            time.sleep(NCBI_RATE)
            handle = Entrez.efetch(
                db="pubmed", id=",".join(idlist),
                rettype="xml", retmode="xml"
            )
            records = Entrez.read(handle)
            handle.close()

            # Score candidates using fuzzy matching
            best_pmid = None
            best_record = None
            best_combined = 0

            for article in records.get('PubmedArticle', []):
                data = extract_record_data(article)
                # Skip records without abstracts (not useful for our pipeline)
                if not data or not data.get('abstract'):
                    continue

                # Calculate title similarity score
                title_score = fuzz.token_sort_ratio(
                    clean_text_for_matching(title),
                    clean_text_for_matching(data['title'])
                )

                # Calculate author similarity score
                author_score = 100
                if authors:
                    author_score = fuzz.token_sort_ratio(
                        clean_text_for_matching(authors),
                        clean_text_for_matching(data.get('authors', ''))
                    )

                # Combined score: weight title more heavily (70/30)
                combined = title_score * 0.7 + author_score * 0.3

                # Check if this candidate meets thresholds and is best so far
                author_ok = (not authors) or (author_score >= FUZZY_AUTHOR_MIN)
                if title_score >= FUZZY_TITLE_THRESHOLD and author_ok and combined > best_combined:
                    best_pmid = data['pmid']
                    best_record = data
                    best_combined = combined

            if best_pmid:
                return best_pmid, best_record

        except Exception:
            continue

    return None, None


# =============================================================================
# Print summary of available API functions
# =============================================================================
print("API functions defined (enhanced with fuzzy matching):")
print("  - get_doi_from_crossref() - CrossRef search with fuzzy matching + short-title fallback")
print("  - fetch_crossref_abstract() - CrossRef abstract fetch (for PubMed fallback)")
print("  - get_pmid_from_doi() - DOI to PMID lookup")
print("  - fetch_abstracts_batch() - Batch PubMed fetch")
print("  - search_pubmed_by_title() - PubMed direct title search (fallback)")
print(f"  - Fuzzy threshold: {FUZZY_TITLE_THRESHOLD}% title, {FUZZY_AUTHOR_MIN}% author (when provided)")

API functions defined (enhanced with fuzzy matching):
  • get_doi_from_crossref() - CrossRef search with fuzzy matching + short-title fallback
  • fetch_crossref_abstract() - CrossRef abstract fetch (for PubMed fallback)
  • get_pmid_from_doi() - DOI to PMID lookup
  • fetch_abstracts_batch() - Batch PubMed fetch
  • search_pubmed_by_title() - PubMed direct title search (fallback)
  • Fuzzy threshold: 75% title, 30% author (when provided)


---
## 5. Phase 2: CrossRef DOI Lookup
Query CrossRef API to find DOIs for references without direct identifiers.

In [ ]:
# Phase 2 setup: Identify references requiring CrossRef DOI lookup.
# Calculates time estimate based on API rate limits and prepares the subset
# of references that do not have DOIs from direct extraction.

print("PHASE 2: CrossRef DOI Lookup")
print("=" * 60)

# ─────────────────────────────────────────────────────────────────────────────
# Filter to references that still need DOI lookup
# These are references where Phase 1 direct extraction found nothing
# ─────────────────────────────────────────────────────────────────────────────
refs_need_crossref = unique_refs[unique_refs['final_doi'].isna()].copy()
print(f"References needing CrossRef: {len(refs_need_crossref):,}")

# ─────────────────────────────────────────────────────────────────────────────
# Estimate processing time based on API rate limits
# CrossRef rate: ~1 request per CROSSREF_RATE seconds
# ─────────────────────────────────────────────────────────────────────────────
est_hours = len(refs_need_crossref) * CROSSREF_RATE / 3600
print(f"Estimated time: ~{est_hours:.1f} hours")

# ─────────────────────────────────────────────────────────────────────────────
# Prepare the processing queue
# ─────────────────────────────────────────────────────────────────────────────
refs_to_process = refs_need_crossref.copy()
print(f"References to process: {len(refs_to_process):,}")

PHASE 2: CrossRef DOI Lookup
References needing CrossRef: 14,974
Estimated time: ~2.1 hours
References to process: 14,974


In [ ]:
# Phase 2 execution: Query CrossRef API to find DOIs for unmatched references.
# Uses fuzzy matching to validate results and saves progress to allow resumption.

results = []
start_time = time.time()
matched_count = 0

# ─────────────────────────────────────────────────────────────────────────────
# Main CrossRef lookup loop with progress tracking
# ─────────────────────────────────────────────────────────────────────────────
for idx, (_, row) in enumerate(tqdm(refs_to_process.iterrows(), total=len(refs_to_process), desc="CrossRef")):
    # Build clean reference text from title, authors, year
    ref_text = clean_reference(f"{row['title']} {row['authors']} {row['year']}")
    
    # Query CrossRef with fuzzy matching validation
    crossref_doi, match_method, title_score, author_score = get_doi_from_crossref(
        ref_text, title=row['title'], authors=row.get('authors', '')
    )
    
    # Rate limiting to respect CrossRef API guidelines
    time.sleep(CROSSREF_RATE)
    
    # ─────────────────────────────────────────────────────────────────────────
    # Store result with match metadata for later analysis
    # ─────────────────────────────────────────────────────────────────────────
    results.append({
        'study_id': row['study_id'],
        'category': row['category'],
        'original_title': row['title'],
        'original_authors': row['authors'],
        'original_year': row['year'],
        'crossref_doi': crossref_doi,
        'match_method': match_method or 'no_match',
        'title_score': title_score,
        'author_score': author_score
    })
    
    # Track successful matches
    if crossref_doi:
        matched_count += 1
    
    # ─────────────────────────────────────────────────────────────────────────
    # Progress reporting every 100 references
    # ─────────────────────────────────────────────────────────────────────────
    if (idx + 1) % 100 == 0:
        elapsed = time.time() - start_time
        rate = (idx + 1) / elapsed * 3600  # References per hour
        print(f"\n[{idx+1:,}/{len(refs_to_process):,}] Match rate: {matched_count/(idx+1)*100:.1f}% ({rate:.0f}/hr)")

# ─────────────────────────────────────────────────────────────────────────────
# Save progress to CSV for resumption capability
# ─────────────────────────────────────────────────────────────────────────────
crossref_results_df = pd.DataFrame(results)
crossref_results_df.to_csv(PROGRESS_CSV, index=False)

print(f"\n✓ CrossRef complete: {matched_count:,} DOIs found ({matched_count/len(refs_to_process)*100:.1f}%)")

CrossRef:   0%|          | 0/14974 [00:00<?, ?it/s]


[100/14,974] Match rate: 48.0% (2273/hr)

[200/14,974] Match rate: 60.5% (2404/hr)

[300/14,974] Match rate: 67.7% (2469/hr)

[400/14,974] Match rate: 72.2% (2400/hr)

[500/14,974] Match rate: 74.8% (2191/hr)

[600/14,974] Match rate: 76.2% (2055/hr)

[700/14,974] Match rate: 77.4% (1981/hr)

[800/14,974] Match rate: 77.5% (1956/hr)

[900/14,974] Match rate: 77.9% (1910/hr)

[1,000/14,974] Match rate: 77.7% (1849/hr)

[1,100/14,974] Match rate: 74.3% (1753/hr)

[1,200/14,974] Match rate: 72.4% (1721/hr)

[1,300/14,974] Match rate: 72.2% (1681/hr)

[1,400/14,974] Match rate: 73.1% (1627/hr)

[1,500/14,974] Match rate: 70.5% (1537/hr)

[1,600/14,974] Match rate: 70.2% (1520/hr)

[1,700/14,974] Match rate: 70.2% (1501/hr)

[1,800/14,974] Match rate: 70.7% (1488/hr)

[1,900/14,974] Match rate: 71.2% (1481/hr)

[2,000/14,974] Match rate: 71.0% (1464/hr)

[2,100/14,974] Match rate: 69.7% (1451/hr)

[2,200/14,974] Match rate: 68.2% (1443/hr)

[2,300/14,974] Match rate: 67.9% (1433/hr)

[2,40

---
## 6. Phase 3: DOI to PMID Conversion
Convert DOIs to PubMed identifiers for abstract fetching.

In [ ]:
# Phase 3: Convert DOIs to PubMed identifiers (PMIDs).
# Merges CrossRef-discovered DOIs with directly extracted DOIs, then queries
# PubMed to retrieve PMIDs for subsequent abstract fetching.

print("PHASE 3: DOI → PMID Conversion")
print("=" * 60)

# ─────────────────────────────────────────────────────────────────────────────
# Rebuild unique_refs with fresh deduplication (in case of notebook re-run)
# ─────────────────────────────────────────────────────────────────────────────
unique_refs_fresh = refs_df.drop_duplicates(subset='signature').copy()

# Re-apply direct DOI extraction
unique_refs_fresh['full_ref'] = (
    unique_refs_fresh['title'].fillna('') + ' ' + 
    unique_refs_fresh['authors'].fillna('') + ' ' +
    unique_refs_fresh['year'].fillna('').astype(str)
)
unique_refs_fresh['extracted_doi'] = unique_refs_fresh['full_ref'].apply(extract_doi)
unique_refs_fresh['final_doi'] = unique_refs_fresh['extracted_doi']

# ─────────────────────────────────────────────────────────────────────────────
# Load CrossRef results and merge DOIs discovered in Phase 2
# ─────────────────────────────────────────────────────────────────────────────
crossref_results = pd.read_csv(PROGRESS_CSV) if PROGRESS_CSV.exists() else pd.DataFrame()

if len(crossref_results) > 0:
    # Create normalized titles for matching
    crossref_results['title_normalized'] = crossref_results['original_title'].str.lower().str.strip()
    unique_refs_fresh['title_normalized'] = unique_refs_fresh['title'].str.lower().str.strip()
    
    # Merge CrossRef DOIs by normalized title
    crossref_doi_map = crossref_results[['title_normalized', 'crossref_doi']].drop_duplicates()
    unique_refs_fresh = unique_refs_fresh.merge(
        crossref_doi_map,
        on='title_normalized',
        how='left'
    )
    # Combine: use direct extraction first, fall back to CrossRef
    unique_refs_fresh['final_doi'] = unique_refs_fresh['final_doi'].combine_first(unique_refs_fresh['crossref_doi'])

unique_refs = unique_refs_fresh

# ─────────────────────────────────────────────────────────────────────────────
# Collect all unique DOIs for PMID lookup
# ─────────────────────────────────────────────────────────────────────────────
all_dois = unique_refs[unique_refs['final_doi'].notna()]['final_doi'].str.lower().unique()
print(f"Total unique DOIs: {len(all_dois):,}")

DOI_PMID_CACHE = DATA_DIR / "doi_pmid_cache.csv"

# ─────────────────────────────────────────────────────────────────────────────
# Query PubMed to convert DOIs to PMIDs
# Uses NCBI Entrez esearch with DOI field
# ─────────────────────────────────────────────────────────────────────────────
doi_to_pmid = {}
print(f"DOIs to look up: {len(all_dois):,}")

if len(all_dois) > 0:
    print("Converting DOIs to PMIDs...")
    new_mappings = 0
    
    for doi in tqdm(all_dois, desc="DOI→PMID"):
        # Query PubMed for PMID using DOI
        pmid = get_pmid_from_doi(doi, Entrez.api_key)
        if pmid:
            doi_to_pmid[doi.lower()] = pmid
            new_mappings += 1
        else:
            doi_to_pmid[doi.lower()] = None  # Cache negative results too
        
        # Rate limiting for NCBI API
        time.sleep(NCBI_RATE)
    
    # ─────────────────────────────────────────────────────────────────────────
    # Save DOI→PMID cache for future runs
    # ─────────────────────────────────────────────────────────────────────────
    cache_rows = [{'doi': k, 'pmid': v if v else 'NO_PMID'} for k, v in doi_to_pmid.items()]
    cache_df = pd.DataFrame(cache_rows)
    cache_df.to_csv(DOI_PMID_CACHE, index=False)
    print(f"\n✓ Found {new_mappings:,} PMIDs, saved to {DOI_PMID_CACHE.name}")

# ─────────────────────────────────────────────────────────────────────────────
# Report conversion results
# ─────────────────────────────────────────────────────────────────────────────
pmids_for_current = sum(1 for d in all_dois if doi_to_pmid.get(d.lower()) is not None)
print(f"\n✓ PMIDs found: {pmids_for_current:,} / {len(all_dois):,} ({pmids_for_current/len(all_dois)*100:.1f}%)")

PHASE 3: DOI → PMID Conversion
Total unique DOIs: 10,525
DOIs to look up: 10,525
Converting DOIs to PMIDs...


DOI→PMID:   0%|          | 0/10525 [00:00<?, ?it/s]


✓ Found 8,223 PMIDs, saved to doi_pmid_cache.csv

✓ PMIDs found: 8,223 / 10,525 (78.1%)


---
## 7. Phase 4: Batch Fetch Abstracts
Fetch PubMed abstracts in batches, with CrossRef fallback and PubMed title search.

In [ ]:
# Phase 4: Batch fetch abstracts from PubMed using collected PMIDs.
# Uses efficient batch requests (200 per call) with retry logic for robustness.

print("PHASE 4: Fetch Abstracts")
print("=" * 60)

# ─────────────────────────────────────────────────────────────────────────────
# Map DOI→PMID conversions to the unique references dataframe
# ─────────────────────────────────────────────────────────────────────────────
unique_refs['doi_lower'] = unique_refs['final_doi'].str.lower()
unique_refs['pmid_from_doi'] = unique_refs['doi_lower'].map(doi_to_pmid)

# Combine PMIDs: use direct extraction first, then DOI→PMID mapping
if 'final_pmid' not in unique_refs.columns:
    unique_refs['final_pmid'] = None
unique_refs['final_pmid'] = unique_refs['final_pmid'].combine_first(unique_refs['pmid_from_doi'])

# ─────────────────────────────────────────────────────────────────────────────
# Collect all unique PMIDs for batch fetching
# ─────────────────────────────────────────────────────────────────────────────
all_pmids = unique_refs[unique_refs['final_pmid'].notna()]['final_pmid'].unique()
print(f"Total unique PMIDs: {len(all_pmids):,}")

# ─────────────────────────────────────────────────────────────────────────────
# Batch fetch PubMed records (200 at a time for efficiency)
# ─────────────────────────────────────────────────────────────────────────────
print("Fetching abstracts in batches...")
start = time.time()
abstract_records, failed_batches = fetch_abstracts_batch(all_pmids, batch_size=200)
elapsed = time.time() - start

# ─────────────────────────────────────────────────────────────────────────────
# Report fetching results
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n✓ Fetched {len(abstract_records):,} records in {elapsed:.0f}s")
with_abstract = sum(1 for r in abstract_records.values() if r.get('abstract'))
print(f"  With abstracts: {with_abstract:,} ({with_abstract/len(abstract_records)*100:.1f}%)")
if failed_batches:
    print(f"  Failed batches: {len(failed_batches):,}")

PHASE 4: Fetch Abstracts
Total unique PMIDs: 8,215
Fetching abstracts in batches...
Batch 37 attempt 1 failed: IncompleteRead(1137 bytes read), retrying...

✓ Fetched 8,209 records in 121s
  With abstracts: 7,750 (94.4%)


In [ ]:
# Phase 4b: CrossRef abstract fallback and results compilation.
# For papers with DOI but no PubMed abstract, attempts to fetch from CrossRef.
# Then compiles all results by re-expanding unique signatures to all original
# (study_id, review_doi) pairs for complete coverage.

print("PHASE 4b: CrossRef Abstract Fallback")
print("=" * 60)

# ─────────────────────────────────────────────────────────────────────────────
# Identify papers that have DOI but no abstract from PubMed
# These can be fetched from CrossRef as a fallback source
# ─────────────────────────────────────────────────────────────────────────────
dois_needing_fallback = []
for _, row in unique_refs.iterrows():
    doi = str(row.get('final_doi', '')).lower() if pd.notna(row.get('final_doi')) else None
    if not doi or doi == 'nan':
        continue
    
    # Check if we got an abstract from PubMed for this reference
    pmid = str(row.get('final_pmid')) if pd.notna(row.get('final_pmid')) else None
    record = abstract_records.get(pmid, {}) if pmid else {}
    
    # If no abstract, add to CrossRef fallback queue
    if not record.get('abstract'):
        dois_needing_fallback.append(doi)

# Deduplicate DOIs
dois_needing_fallback = list(set(dois_needing_fallback))
print(f"DOIs needing CrossRef abstract lookup: {len(dois_needing_fallback):,}")

# ─────────────────────────────────────────────────────────────────────────────
# Fetch abstracts from CrossRef for papers missing PubMed abstracts
# ─────────────────────────────────────────────────────────────────────────────
doi_to_crossref_abstract = {}
if dois_needing_fallback:
    print("Fetching abstracts from CrossRef...")
    for doi in tqdm(dois_needing_fallback, desc="CrossRef abstracts"):
        abstract = fetch_crossref_abstract(doi)
        # Only keep abstracts with meaningful content (>50 chars)
        if abstract and len(abstract) > 50:
            doi_to_crossref_abstract[doi.lower()] = abstract
        time.sleep(0.2)  # Rate limiting
    
    print(f"\n✓ Found {len(doi_to_crossref_abstract):,} abstracts from CrossRef ({len(doi_to_crossref_abstract)/len(dois_needing_fallback)*100:.1f}%)")
else:
    print("✓ No references need CrossRef abstract fallback")

# ─────────────────────────────────────────────────────────────────────────────
# COMPILE FINAL RESULTS
# Build a lookup dictionary from signature → abstract data
# ─────────────────────────────────────────────────────────────────────────────
print("\nCompiling final results...")
print("=" * 60)

signature_to_abstract = {}

for _, row in unique_refs.iterrows():
    sig = row['signature']
    pmid = str(row['final_pmid']) if pd.notna(row['final_pmid']) else None
    record = abstract_records.get(pmid, {}) if pmid else {}
    doi = str(row['final_doi']).lower() if pd.notna(row['final_doi']) else None
    
    # Priority: PubMed abstract > CrossRef abstract
    abstract = record.get('abstract', '')
    abstract_source = 'pubmed' if abstract else None
    
    # Fallback to CrossRef abstract if PubMed had none
    if not abstract and doi and doi in doi_to_crossref_abstract:
        abstract = doi_to_crossref_abstract[doi]
        abstract_source = 'crossref'
    
    # Determine match method based on how we found the identifier
    if pd.notna(row.get('extracted_pmid')):
        method = 'pmid_direct'  # PMID in reference text
    elif pd.notna(row.get('extracted_doi')) or pd.notna(row.get('ref_doi')):
        method = 'doi_direct'   # DOI in reference text
    elif pd.notna(row.get('crossref_doi')):
        method = 'crossref'     # DOI from CrossRef lookup
    else:
        method = 'no_match'
    
    # Store compiled data for this signature
    signature_to_abstract[sig] = {
        'pmid': pmid,
        'doi': row['final_doi'],
        'matched_title': record.get('title', ''),
        'matched_authors': record.get('authors', ''),
        'matched_year': record.get('year', ''),
        'abstract': abstract,
        'abstract_source': abstract_source,
        'match_method': method if pmid or (doi and abstract) else 'no_match'
    }

print(f"Abstract data built for {len(signature_to_abstract):,} unique signatures")

# ─────────────────────────────────────────────────────────────────────────────
# Report abstract source breakdown
# ─────────────────────────────────────────────────────────────────────────────
pubmed_abstracts = sum(1 for v in signature_to_abstract.values() if v.get('abstract_source') == 'pubmed')
crossref_abstracts_found = sum(1 for v in signature_to_abstract.values() if v.get('abstract_source') == 'crossref')
print(f"  Abstracts from PubMed: {pubmed_abstracts:,}")
print(f"  Abstracts from CrossRef (fallback): {crossref_abstracts_found:,}")

# ─────────────────────────────────────────────────────────────────────────────
# EXPAND RESULTS: Map signature data back to all original references
# A single signature may appear in multiple reviews, so we re-expand
# ─────────────────────────────────────────────────────────────────────────────
output_rows = []

for _, row in refs_df.iterrows():
    sig = row['signature']
    abstract_data = signature_to_abstract.get(sig, {})
    
    # Preserve original reference metadata plus matched data
    output_rows.append({
        'study_id': row['study_id'],
        'review_doi': row['review_doi'],       # Key for ground truth: which review cited this
        'category': row['category'],            # included/excluded/awaiting/ongoing
        'original_title': row['title'],
        'original_authors': row['authors'],
        'original_year': row['year'],
        'pmid': abstract_data.get('pmid'),
        'doi': abstract_data.get('doi'),
        'matched_title': abstract_data.get('matched_title', ''),
        'matched_authors': abstract_data.get('matched_authors', ''),
        'matched_year': abstract_data.get('matched_year', ''),
        'abstract': abstract_data.get('abstract', ''),
        'abstract_source': abstract_data.get('abstract_source', ''),
        'match_method': abstract_data.get('match_method', 'no_match')
    })

results_df = pd.DataFrame(output_rows)

# ─────────────────────────────────────────────────────────────────────────────
# Report compilation statistics
# ─────────────────────────────────────────────────────────────────────────────
print(f"\nTotal references: {len(results_df):,}")
print(f"Unique (study_id, review_doi) pairs: {results_df.groupby(['study_id', 'review_doi']).ngroups:,}")
print(f"\nMatch methods:")
print(results_df['match_method'].value_counts().to_string())

PHASE 4b: CrossRef Abstract Fallback
DOIs needing CrossRef abstract lookup: 2,767
Fetching abstracts from CrossRef...


CrossRef abstracts:   0%|          | 0/2767 [00:00<?, ?it/s]


✓ Found 640 abstracts from CrossRef (23.1%)

Compiling final results...
Abstract data built for 15,712 unique signatures
  Abstracts from PubMed: 8,031
  Abstracts from CrossRef (fallback): 663

Total references: 17,368
Unique (study_id, review_doi) pairs: 12,665

Match methods:
match_method
crossref      10301
no_match       7054
doi_direct       13


In [ ]:
# Phase 4c: PubMed direct title search for remaining unmatched references.
# Searches PubMed directly using reference titles with fuzzy matching validation
# to catch papers that exist in PubMed but were missed by DOI-based lookups.

print("PHASE 4c: PubMed Direct Title Search")
print("=" * 60)

# ─────────────────────────────────────────────────────────────────────────────
# Identify signatures that still lack abstracts after all previous phases
# ─────────────────────────────────────────────────────────────────────────────
sigs_needing_search = []
for sig, data in signature_to_abstract.items():
    if not data.get('abstract') or data.get('match_method') == 'no_match':
        sigs_needing_search.append(sig)

print(f"References without abstracts: {len(sigs_needing_search):,}")

# ─────────────────────────────────────────────────────────────────────────────
# Search PubMed by title for remaining unmatched references
# Uses fuzzy matching to validate results
# ─────────────────────────────────────────────────────────────────────────────
found = 0

for sig in tqdm(sigs_needing_search, desc="PubMed title search"):
    # Get the original reference data for this signature
    matching = unique_refs[unique_refs['signature'] == sig]
    if matching.empty:
        continue
    ref_row = matching.iloc[0]
    
    # Extract search parameters
    title = ref_row.get('title', '')
    if pd.isna(title) or not title:
        continue
    authors = ref_row.get('authors', '')
    year = ref_row.get('year', '')

    # Search PubMed directly by title
    pmid, record = search_pubmed_by_title(title, authors=authors, year=year)

    # If found with valid abstract, update the signature data
    if pmid and record and record.get('abstract'):
        found += 1

        signature_to_abstract[sig] = {
            'pmid': pmid,
            'doi': record.get('doi', ''),
            'matched_title': record.get('title', ''),
            'matched_authors': record.get('authors', ''),
            'matched_year': record.get('year', ''),
            'abstract': record['abstract'],
            'abstract_source': 'pubmed_title_search',
            'match_method': 'pubmed_title_search'
        }

print(f"\n✓ Found {found:,} additional abstracts via PubMed title search")

# ─────────────────────────────────────────────────────────────────────────────
# If we found more abstracts, rebuild the full results dataframe
# ─────────────────────────────────────────────────────────────────────────────
if found > 0:
    output_rows = []
    for _, row in refs_df.iterrows():
        sig = row['signature']
        abstract_data = signature_to_abstract.get(sig, {})
        output_rows.append({
            'study_id': row['study_id'],
            'review_doi': row['review_doi'],
            'category': row['category'],
            'original_title': row['title'],
            'original_authors': row['authors'],
            'original_year': row['year'],
            'pmid': abstract_data.get('pmid'),
            'doi': abstract_data.get('doi'),
            'matched_title': abstract_data.get('matched_title', ''),
            'matched_authors': abstract_data.get('matched_authors', ''),
            'matched_year': abstract_data.get('matched_year', ''),
            'abstract': abstract_data.get('abstract', ''),
            'abstract_source': abstract_data.get('abstract_source', ''),
            'match_method': abstract_data.get('match_method', 'no_match')
        })
    results_df = pd.DataFrame(output_rows)
    print(f"  Results rebuilt: {len(results_df):,} rows")

# ─────────────────────────────────────────────────────────────────────────────
# Report updated match methods after title search
# ─────────────────────────────────────────────────────────────────────────────
print(f"\nUpdated match methods:")
print(results_df['match_method'].value_counts().to_string())

PHASE 4c: PubMed Direct Title Search
References without abstracts: 6,412


PubMed title search:   0%|          | 0/6412 [00:00<?, ?it/s]


✓ Found 428 additional abstracts via PubMed title search
  Results rebuilt: 17,368 rows

Updated match methods:
match_method
crossref               10277
no_match                5933
pubmed_title_search     1145
doi_direct                13


In [ ]:
# Save final output with matched references and their abstracts.
# Filters to successfully matched references and preserves review_doi for
# downstream ground truth construction.

# ─────────────────────────────────────────────────────────────────────────────
# Filter to successfully matched references only
# Unmatched references are excluded from the output
# ─────────────────────────────────────────────────────────────────────────────
matched_refs = results_df[results_df['match_method'] != 'no_match'].copy()

# ─────────────────────────────────────────────────────────────────────────────
# Report final matching statistics
# ─────────────────────────────────────────────────────────────────────────────
print("FINAL RESULTS")
print("=" * 60)
print(f"Total matched: {len(matched_refs):,} / {len(results_df):,} ({len(matched_refs)/len(results_df)*100:.1f}%)")
print(f"Unique (study_id, review_doi) pairs: {matched_refs.groupby(['study_id', 'review_doi']).ngroups:,}")

# Breakdown by reference category
print(f"\nBy category:")
print(matched_refs['category'].value_counts().to_string())

# Abstract coverage statistics
has_abstract = matched_refs['abstract'].str.len() > 0
print(f"\nWith abstracts: {has_abstract.sum():,} ({has_abstract.mean()*100:.1f}%)")

# ─────────────────────────────────────────────────────────────────────────────
# Save to CSV for use in downstream notebooks (ground truth construction)
# Key columns preserved: review_doi (for linking to reviews), category, abstract
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n✓ Columns in output: {matched_refs.columns.tolist()}")

matched_refs.to_csv(OUTPUT_CSV, index=False)
print(f"\n✓ Saved to {OUTPUT_CSV}")

FINAL RESULTS
Total matched: 11,435 / 17,368 (65.8%)
Unique (study_id, review_doi) pairs: 9,176

By category:
category
excluded          4529
additional        3815
included          2658
awaiting           241
ongoing            142
other_versions      50

With abstracts: 10,945 (95.7%)

✓ Columns in output: ['study_id', 'review_doi', 'category', 'original_title', 'original_authors', 'original_year', 'pmid', 'doi', 'matched_title', 'matched_authors', 'matched_year', 'abstract', 'abstract_source', 'match_method']

✓ Saved to c:\Users\juanx\Documents\LSE-UKHSA Project\Data\referenced_paper_abstracts.csv


---
## 8. Save Results and Summary
Export matched references with abstracts and display pipeline statistics.

In [ ]:
# Pipeline summary statistics showing results from each phase.
# Provides visibility into match rates, abstract coverage, and data sources.

print("=" * 60)
print("PIPELINE SUMMARY")
print("=" * 60)

# ─────────────────────────────────────────────────────────────────────────────
# Input statistics
# ─────────────────────────────────────────────────────────────────────────────
print(f"\nInput: {len(unique_refs):,} unique references from public health reviews")

# ─────────────────────────────────────────────────────────────────────────────
# Phase 1: Direct identifier extraction from reference text
# ─────────────────────────────────────────────────────────────────────────────
print(f"\nPhase 1 - Direct extraction:")
direct_doi = unique_refs['extracted_doi'].notna().sum() if 'extracted_doi' in unique_refs.columns else 0
print(f"  DOIs extracted: {direct_doi:,}")

# ─────────────────────────────────────────────────────────────────────────────
# Phase 2: CrossRef API lookups with fuzzy matching validation
# ─────────────────────────────────────────────────────────────────────────────
if PROGRESS_CSV.exists():
    crossref_df = pd.read_csv(PROGRESS_CSV)
    crossref_found = crossref_df['crossref_doi'].notna().sum()
    print(f"\nPhase 2 - CrossRef DOI lookup (with fuzzy matching):")
    print(f"  DOIs found: {crossref_found:,} / {len(crossref_df):,} ({crossref_found/len(crossref_df)*100:.1f}%)")

# ─────────────────────────────────────────────────────────────────────────────
# Phase 3: DOI to PMID conversion via PubMed
# ─────────────────────────────────────────────────────────────────────────────
print(f"\nPhase 3 - DOI → PMID:")
actual_pmids = sum(1 for v in doi_to_pmid.values() if v is not None)
print(f"  PMIDs found: {actual_pmids:,}")

# ─────────────────────────────────────────────────────────────────────────────
# Phase 4: Batch abstract fetching from PubMed
# ─────────────────────────────────────────────────────────────────────────────
print(f"\nPhase 4 - Abstract fetch:")
print(f"  PubMed records fetched: {len(abstract_records):,}")
pubmed_with_abstract = sum(1 for r in abstract_records.values() if r.get('abstract'))
print(f"  With PubMed abstracts: {pubmed_with_abstract:,}")

# ─────────────────────────────────────────────────────────────────────────────
# Phase 4b: CrossRef abstract fallback for DOIs without PubMed abstracts
# ─────────────────────────────────────────────────────────────────────────────
print(f"\nPhase 4b - CrossRef abstract fallback:")
print(f"  Additional abstracts from CrossRef: {len(doi_to_crossref_abstract):,}")

# ─────────────────────────────────────────────────────────────────────────────
# Phase 4c: PubMed direct title search for remaining unmatched
# ─────────────────────────────────────────────────────────────────────────────
pubmed_title_count = sum(1 for v in signature_to_abstract.values() if v.get('abstract_source') == 'pubmed_title_search')
print(f"\nPhase 4c - PubMed direct title search:")
print(f"  Additional abstracts from title search: {pubmed_title_count:,}")

# ─────────────────────────────────────────────────────────────────────────────
# Total abstract recovery across all sources
# ─────────────────────────────────────────────────────────────────────────────
total_abstracts = pubmed_with_abstract + len(doi_to_crossref_abstract) + pubmed_title_count
print(f"\nTotal abstracts recovered: {total_abstracts:,}")

# ─────────────────────────────────────────────────────────────────────────────
# Final output statistics
# ─────────────────────────────────────────────────────────────────────────────
print(f"\nFinal output: {OUTPUT_CSV.name}")
print(f"  Matched: {len(matched_refs):,} ({len(matched_refs)/len(unique_refs)*100:.1f}%)")
print(f"  By category: {dict(matched_refs['category'].value_counts())}")

# Abstract coverage breakdown
with_abstract = matched_refs['abstract'].str.len() > 0 
print(f"\nAbstract coverage:")
print(f"  References with abstracts: {with_abstract.sum():,} / {len(matched_refs):,} ({with_abstract.mean()*100:.1f}%)")
if 'abstract_source' in matched_refs.columns:
    print(f"  Source breakdown: {dict(matched_refs[matched_refs['abstract'].str.len() > 0]['abstract_source'].value_counts())}")

PIPELINE SUMMARY

Input: 15,795 unique references from public health reviews

Phase 1 - Direct extraction:
  DOIs extracted: 2

Phase 2 - CrossRef DOI lookup (with fuzzy matching):
  DOIs found: 10,929 / 14,974 (73.0%)

Phase 3 - DOI → PMID:
  PMIDs found: 8,223

Phase 4 - Abstract fetch:
  PubMed records fetched: 8,209
  With PubMed abstracts: 7,750

Phase 4b - CrossRef abstract fallback:
  Additional abstracts from CrossRef: 640

Phase 4c - PubMed direct title search:
  Additional abstracts from title search: 1,034

Total abstracts recovered: 9,424

Final output: referenced_paper_abstracts.csv
  Matched: 11,435 (72.4%)
  By category: {'excluded': np.int64(4529), 'additional': np.int64(3815), 'included': np.int64(2658), 'awaiting': np.int64(241), 'ongoing': np.int64(142), 'other_versions': np.int64(50)}

Abstract coverage:
  References with abstracts: 10,945 / 11,435 (95.7%)
  Source breakdown: {'pubmed': np.int64(9097), 'pubmed_title_search': np.int64(1145), 'crossref': np.int64(703)